# Europe wetlands data preparation.  
This notebook loads local wetland and RAMSAR types data for countries, river basins and protected areas in Europe (from local, due to size and reading times) and prepares them for indicator-data seeding format.  
Data is available in the project's bucket.

In [1]:
import pandas as pd
import geopandas as gpd
import json
import os


In [2]:
RAW_DATA_FOLDER = '../data/raw/European Wetlands Atlas/'
VECTOR_DATA_FOLDER = '../data/raw/vectors/'

countries_data_file = 'Europe_buffered_ha_rounded.gpkg'
countries_locations_file = 'europe_country_boundaries.geojson'


In [3]:
label_dict = {
    "ha_1": "rice",
    "ha_2": "rip_broad",
    "ha_3": "riparian",
    "ha_4": "rip_conif",
    "ha_6": "rip_mixed",
    "ha_8": "wet_meadow",
    "ha_9": "wet_grass",
    "ha_10": "wet_heath",
    "ha_11": "riv_scrub",
    "ha_12": "beach_dune",
    "ha_13": "inl_marsh",
    "ha_14": "open_mire",
    "ha_15": "salt_marsh",
    "ha_16": "cst_lagoon",
    "ha_17": "estuary",
    "ha_18": "saltpan",
    "ha_19": "intertidal",
    "ha_20": "watercourse",
    "ha_21": "lake_pond",
    "ha_22": "marine_low",
    "ha_23": "reservoir",
    "ha_24": "uncert_wet"
}
ramsar_dict = {
    "ha_marine_coastal": "coastal",
    "ha_inland": "inland",
    "ha_man_made": "manmade"
}

## Country data

In [4]:
#Load country locations
country_locations = gpd.read_file(os.path.join(VECTOR_DATA_FOLDER, countries_locations_file))
country_locations = country_locations[['id', 'name', 'type', 'code']]

Skipping field bbox: unsupported OGR type: 3


In [5]:
country_df = gpd.read_file(os.path.join(RAW_DATA_FOLDER, countries_data_file))
country_df.columns = country_df.columns.str.lower()
country_df.drop(columns=['geometry'], inplace=True)
# keep only name and fields starting with ha
country_df = country_df[['name_engl'] + [col for col in country_df.columns if col.startswith('ha')]]
country_df.head()

,name_engl,ha_1,ha_2,ha_3,ha_4,ha_6,ha_8,ha_9,ha_10,ha_11,...,ha_19,ha_20,ha_21,ha_22,ha_23,ha_total,ha_marine_coastal,ha_inland,ha_man_made,ha_total_3classes
0,Lithuania,0.00,7777.26,62263.53,6896.88,7826.85,49965.48,390.51,9.72,0.00,...,82.17,11421.36,109228.14,8142.75,5606.10,366706.71,49654.17,261473.22,55579.32,366706.71
1,Poland,0.00,53772.21,64386.81,31330.89,26488.35,582062.31,710.10,151.11,0.00,...,200.34,71519.67,375915.69,41492.52,22783.14,1526397.21,121678.38,735429.69,669289.14,1526397.21
2,Greece,37071.45,5506.38,94519.44,1723.05,2059.92,5086.80,2973.42,3.78,23.49,...,3571.65,16043.31,69268.68,346379.13,26031.42,745304.22,455857.20,219153.60,70293.42,745304.22
3,Albania,0.00,3972.60,32197.68,497.34,438.84,1090.89,478.98,9.00,0.00,...,1920.78,4858.92,37256.31,32656.32,9332.91,164030.85,59247.45,94315.05,10468.35,164030.85
4,Portugal,42273.09,9628.02,104290.83,2534.85,4574.97,5569.92,1126.44,300.15,158.04,...,3341.43,14489.91,23302.89,37988.73,42876.63,385857.81,126556.56,162192.51,97108.74,385857.81


### Wetland types

In [6]:
country_totals = country_df[['name_engl', 'ha_total']]
country_df_long = country_df.drop(columns=['ha_total', 'ha_marine_coastal', 'ha_inland', 'ha_man_made', 'ha_total_3classes']).copy()
country_df_long = country_df_long.melt(id_vars=['name_engl'], var_name='label', value_name='value')
country_df_long['label'] = country_df_long['label'].map(label_dict)
country_df_long = country_df_long.merge(country_totals, on='name_engl', how='left')
country_df_long['value'] = (country_df_long['value'] / country_df_long['ha_total']) * 100
country_df_long.drop(columns=['ha_total'], inplace=True)
country_df_long = country_df_long.join(country_locations.set_index('name'), on='name_engl', how='left')
country_df_long.rename(columns={'name_engl': 'name'}, inplace=True)
country_df_long = country_df_long[['id', 'label', 'value']]
print(f'Total entries: {len(country_df_long)}')
country_df_long = country_df_long[country_df_long['value'] > 0]
print(f'Total entries with data: {len(country_df_long)}')
display(country_df_long.head())


Total entries: 798
Total entries with data: 624


,id,label,value
2,europe_adminregion_grc,rice,4.974002
4,europe_adminregion_prt,rice,10.955613
5,europe_adminregion_cze,rice,0.016189
8,europe_adminregion_hun,rice,0.934324
11,europe_adminregion_fra,rice,1.537523


In [7]:
sel_country = 'europe_adminregion_esp'
country_df_long[country_df_long['id'] == sel_country]['value'].sum()

np.float64(100.0)

### RAMSAR types

In [8]:
country_df_ramsar = country_df[['name_engl', 'ha_marine_coastal', 'ha_inland', 'ha_man_made']].copy()
ramsar_totals = country_df[['name_engl', 'ha_total_3classes']]
country_df_ramsar_long = country_df_ramsar.melt(id_vars=['name_engl'], var_name='label', value_name='value')
country_df_ramsar_long['label'] = country_df_ramsar_long['label'].map(ramsar_dict)
country_df_ramsar_long = country_df_ramsar_long.merge(ramsar_totals, on='name_engl', how='left')
country_df_ramsar_long['value'] = (country_df_ramsar_long['value'] / country_df_ramsar_long['ha_total_3classes']) * 100
country_df_ramsar_long.drop(columns=['ha_total_3classes'], inplace=True)
country_df_ramsar_long = country_df_ramsar_long.join(country_locations.set_index('name'), on='name_engl', how='left')
country_df_ramsar_long.rename(columns={'name_engl': 'name'}, inplace=True)
country_df_ramsar_long = country_df_ramsar_long[['id', 'label', 'value']]
print(f'Total entries: {len(country_df_ramsar_long)}')
country_df_ramsar_long = country_df_ramsar_long[country_df_ramsar_long['value'] > 0]
print(f'Total entries with data: {len(country_df_ramsar_long)}')
display(country_df_ramsar_long.head())

Total entries: 114
Total entries with data: 101


,id,label,value
0,europe_adminregion_ltu,coastal,13.540568
1,europe_adminregion_pol,coastal,7.971607
2,europe_adminregion_grc,coastal,61.163910
3,europe_adminregion_alb,coastal,36.119699
4,europe_adminregion_prt,coastal,32.798756


In [9]:
sel_country = 'europe_adminregion_esp'
country_df_ramsar_long[country_df_ramsar_long['id'] == sel_country]['value'].sum()

np.float64(100.0)

## Protected areas

In [10]:
pa_data_file = 'protected_areas_name_unique_final.gpkg'
pa_df = gpd.read_file(os.path.join(RAW_DATA_FOLDER, pa_data_file))
pa_df.columns = pa_df.columns.str.lower()
pa_df.drop(columns=['geometry'], inplace=True)
# keep only name and fields starting with ha
pa_df = pa_df[['name_eng'] + [col for col in pa_df.columns if col.startswith('ha')]]
pa_df.head()

,name_eng,ha_1,ha_2,ha_3,ha_4,ha_6,ha_8,ha_9,ha_10,ha_11,...,ha_19,ha_20,ha_21,ha_22,ha_23,ha_total,ha_marine_coastal,ha_inland,ha_man_made,ha_total_3classes
0,NSG Emsaue,0.0,7.11,251.37,0.99,0.18,207.81,0.0,0.0,0.0,...,0.00,0.00,0.0,0.00,0.0,467.46,0.00,259.65,207.81,467.46
1,NSG Lippeaue,0.0,14.67,252.99,5.22,0.18,584.10,0.0,0.0,0.0,...,0.00,16.83,0.0,0.00,0.0,873.99,0.00,289.89,584.10,873.99
2,Mittlere Elbe,0.0,28.44,44.64,0.00,0.00,411.66,0.0,0.0,0.0,...,0.00,9.90,0.0,0.00,0.0,494.64,0.00,82.98,411.66,494.64
3,Aspskär Islands,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,...,5.49,0.00,0.0,143.28,0.0,149.31,149.31,0.00,0.00,149.31
4,Söderskär and Långören Archipelago,0.0,0.00,0.00,2.34,0.00,0.00,0.0,0.0,0.0,...,83.61,0.00,0.0,1830.06,0.0,1918.17,1915.83,2.34,0.00,1918.17


In [11]:
# Same IDs as the Europe protected areas in locations.json (see _new_locations.ipynb): the row index of the file.
# The rows are matched by this ID, as name_eng isn't unique
pa_df['id'] = 'europe_wdpa_' + pa_df.index.astype(str)
pa_df[['name_eng', 'id']].head()

,name_eng,id
0,NSG Emsaue,europe_wdpa_0
1,NSG Lippeaue,europe_wdpa_1
2,Mittlere Elbe,europe_wdpa_2
3,Aspskär Islands,europe_wdpa_3
4,Söderskär and Långören Archipelago,europe_wdpa_4


### Wetland types

In [12]:
pa_totals = pa_df[['id', 'ha_total']]
pa_df_long = pa_df.drop(columns=['name_eng', 'ha_total', 'ha_marine_coastal', 'ha_inland', 'ha_man_made', 'ha_total_3classes']).copy()
pa_df_long = pa_df_long.melt(id_vars=['id'], var_name='label', value_name='value')
pa_df_long['label'] = pa_df_long['label'].map(label_dict)
pa_df_long = pa_df_long.merge(pa_totals, on='id', how='left')
pa_df_long['value'] = (pa_df_long['value'] / pa_df_long['ha_total']) * 100
pa_df_long = pa_df_long[['id', 'label', 'value']]
print(f'Total entries: {len(pa_df_long)}')
pa_df_long = pa_df_long[pa_df_long['value'] > 0]
print(f'Total entries with data: {len(pa_df_long)}')
display(pa_df_long.head())

Total entries: 935403
Total entries with data: 132604


,id,label,value
84,europe_wdpa_84,rice,0.850252
112,europe_wdpa_112,rice,2.516547
142,europe_wdpa_142,rice,26.344080
230,europe_wdpa_230,rice,50.023875
246,europe_wdpa_246,rice,18.433345


### RAMSAR types

In [13]:
pa_df_ramsar = pa_df[['id', 'ha_marine_coastal', 'ha_inland', 'ha_man_made']].copy()
pa_ramsar_totals = pa_df[['id', 'ha_total_3classes']]
pa_df_ramsar_long = pa_df_ramsar.melt(id_vars=['id'], var_name='label', value_name='value')
pa_df_ramsar_long['label'] = pa_df_ramsar_long['label'].map(ramsar_dict)
pa_df_ramsar_long = pa_df_ramsar_long.merge(pa_ramsar_totals, on='id', how='left')
pa_df_ramsar_long['value'] = (pa_df_ramsar_long['value'] / pa_df_ramsar_long['ha_total_3classes']) * 100
pa_df_ramsar_long = pa_df_ramsar_long[['id', 'label', 'value']]
print(f'Total entries: {len(pa_df_ramsar_long)}')
pa_df_ramsar_long = pa_df_ramsar_long[pa_df_ramsar_long['value'] > 0]
print(f'Total entries with data: {len(pa_df_ramsar_long)}')
display(pa_df_ramsar_long.head())

Total entries: 133629
Total entries with data: 63742


,id,label,value
3,europe_wdpa_3,coastal,100.000000
4,europe_wdpa_4,coastal,99.878009
5,europe_wdpa_5,coastal,99.739244
6,europe_wdpa_6,coastal,97.660329
7,europe_wdpa_7,coastal,99.120040


## River basins

In [14]:
basins_data_file = 'River_basins_ha.gpkg'
basin_df = gpd.read_file(os.path.join(RAW_DATA_FOLDER, basins_data_file))
basin_df = basin_df[~basin_df['geometry'].is_empty]
basin_df.columns = basin_df.columns.str.lower()
basin_df.drop(columns=['geometry'], inplace=True)
# The wetland types columns are named ha_c<class> instead of ha_<class>
basin_df.columns = basin_df.columns.str.replace(r'^ha_c(\d+)$', r'ha_\1', regex=True)
# keep only name and fields starting with ha
basin_df = basin_df[['name & country'] + [col for col in basin_df.columns if col.startswith('ha')]]
basin_types_cols = [col for col in basin_df.columns if col in label_dict]
basin_df['ha_total'] = basin_df[basin_types_cols].sum(axis=1)
basin_df['ha_total_3classes'] = basin_df[list(ramsar_dict)].sum(axis=1)
basin_df.head()

,name & country,ha_marine_coastal,ha_inland,ha_man_made,ha_1,ha_2,ha_3,ha_4,ha_6,ha_8,...,ha_16,ha_17,ha_18,ha_19,ha_20,ha_21,ha_22,ha_23,ha_total,ha_total_3classes
0,vistula river basin cross border,64023.21,388821.96,436125.24,0.00,28805.04,873.18,15716.97,12308.85,360915.12,...,30411.72,307.80,60587.46,175.95,57135.33,194242.77,29668.50,14622.66,888970.41,888970.41
1,tornionjoki river basin cross border,5740.02,422897.22,15.12,0.00,10716.21,22146.30,24275.25,26355.06,15.12,...,0.00,2.07,0.00,333.63,8952.66,56764.35,4927.86,0.00,428652.36,428652.36
2,tagus river basin cross border,43111.98,143020.71,75364.02,21532.05,11789.46,95506.65,4496.76,2257.47,6197.22,...,686.52,28274.22,1341.45,1008.90,11540.25,11291.04,8433.63,46293.30,261496.71,261496.71
3,skagerrak and kattegat river basin cross border,91079.19,1605143.07,24801.48,0.00,15714.18,106696.80,161554.68,14541.03,8209.62,...,139.14,4789.98,0.00,2357.28,8209.53,974862.54,81084.42,16591.86,1721023.74,1721023.74
4,seine river basin cross border,122458.50,138661.56,79353.18,6.84,15140.79,69014.70,199.80,293.13,69184.98,...,6.75,5614.47,0.00,55904.94,13652.37,29810.79,51757.56,10161.36,340473.24,340473.24


In [15]:
basin_names = basin_df['name & country'].str.split(', ', expand=True)
basin_names.columns = ['name', 'country']
basin_names['name'] = basin_names['name'].str.title() + ' ' + basin_names['country'].apply(lambda x: f'({x})' if x else '')
basin_names['name'] = basin_names.groupby('name').cumcount().astype(str).radd(basin_names['name'] + ' ')
basin_names['name'] = basin_names['name'].str.replace(' 0', '', regex=False)
basin_names['name'] = basin_names['name'].str.replace('2', '3').str.replace('1', '2')
basin_names['name'] = basin_names['name'].str.strip()
basin_df['id'] = 'europe_hydrobasin_' + basin_names['name'].str.replace(' ', '_').str.lower()
assert basin_df['id'].is_unique, 'Duplicate river basin IDs'
basin_df[['name & country', 'id']].head()

,name & country,id
0,vistula river basin cross border,europe_hydrobasin_vistula_river_basin_cross_bo...
1,tornionjoki river basin cross border,europe_hydrobasin_tornionjoki_river_basin_cros...
2,tagus river basin cross border,europe_hydrobasin_tagus_river_basin_cross_border
3,skagerrak and kattegat river basin cross border,europe_hydrobasin_skagerrak_and_kattegat_river...
4,seine river basin cross border,europe_hydrobasin_seine_river_basin_cross_border


### Wetland types

In [16]:
basin_totals = basin_df[['id', 'ha_total']]
basin_df_long = basin_df.drop(columns=['name & country', 'ha_total', 'ha_marine_coastal', 'ha_inland', 'ha_man_made', 'ha_total_3classes']).copy()
basin_df_long = basin_df_long.melt(id_vars=['id'], var_name='label', value_name='value')
basin_df_long['label'] = basin_df_long['label'].map(label_dict)
basin_df_long = basin_df_long.merge(basin_totals, on='id', how='left')
basin_df_long['value'] = (basin_df_long['value'] / basin_df_long['ha_total']) * 100
basin_df_long = basin_df_long[['id', 'label', 'value']]
print(f'Total entries: {len(basin_df_long)}')
basin_df_long = basin_df_long[basin_df_long['value'] > 0]
print(f'Total entries with data: {len(basin_df_long)}')
display(basin_df_long.head())

Total entries: 4263
Total entries with data: 2674


,id,label,value
2,europe_hydrobasin_tagus_river_basin_cross_border,rice,8.234157
4,europe_hydrobasin_seine_river_basin_cross_border,rice,0.002009
6,europe_hydrobasin_rhone_river_basin_cross_border,rice,5.630280
13,europe_hydrobasin_guadiana_river_basin_cross_b...,rice,13.189521
14,europe_hydrobasin_elbe_river_basin_cross_border,rice,0.002975


In [17]:
sel_basin = basin_df_long['id'].iloc[0]
basin_df_long[basin_df_long['id'] == sel_basin]['value'].sum()

np.float64(100.0)

### RAMSAR types

In [18]:
basin_df_ramsar = basin_df[['id', 'ha_marine_coastal', 'ha_inland', 'ha_man_made']].copy()
basin_ramsar_totals = basin_df[['id', 'ha_total_3classes']]
basin_df_ramsar_long = basin_df_ramsar.melt(id_vars=['id'], var_name='label', value_name='value')
basin_df_ramsar_long['label'] = basin_df_ramsar_long['label'].map(ramsar_dict)
basin_df_ramsar_long = basin_df_ramsar_long.merge(basin_ramsar_totals, on='id', how='left')
basin_df_ramsar_long['value'] = (basin_df_ramsar_long['value'] / basin_df_ramsar_long['ha_total_3classes']) * 100
basin_df_ramsar_long = basin_df_ramsar_long[['id', 'label', 'value']]
print(f'Total entries: {len(basin_df_ramsar_long)}')
basin_df_ramsar_long = basin_df_ramsar_long[basin_df_ramsar_long['value'] > 0]
print(f'Total entries with data: {len(basin_df_ramsar_long)}')
display(basin_df_ramsar_long.head())

Total entries: 609
Total entries with data: 496


,id,label,value
0,europe_hydrobasin_vistula_river_basin_cross_bo...,coastal,7.201951
1,europe_hydrobasin_tornionjoki_river_basin_cros...,coastal,1.339085
2,europe_hydrobasin_tagus_river_basin_cross_border,coastal,16.486624
3,europe_hydrobasin_skagerrak_and_kattegat_river...,coastal,5.292152
4,europe_hydrobasin_seine_river_basin_cross_border,coastal,35.967144


In [19]:
basin_df_ramsar_long[basin_df_ramsar_long['id'] == sel_basin]['value'].sum()

np.float64(99.99999999999999)

## Europe totals

In [20]:
# Total for wetland types
europe_types = country_df.drop(columns=['name_engl', 'ha_total', 'ha_marine_coastal', 'ha_inland', 'ha_man_made', 'ha_total_3classes']).copy()
europe_types = europe_types.melt(id_vars=[], var_name='label', value_name='value')
europe_types = europe_types.groupby('label').sum().reset_index()
europe_types['value'] = (europe_types['value'] / europe_types['value'].sum()) * 100
europe_types['id'] = 'europe_global'
europe_types['label'] = europe_types['label'].map(label_dict)
print(europe_types['value'].sum())
display(europe_types)

100.0


,label,value,id
0,rice,1.040934,europe_global
1,wet_heath,15.070815,europe_global
2,riv_scrub,0.006649,europe_global
3,beach_dune,0.932679,europe_global
4,inl_marsh,1.766624,europe_global
5,open_mire,18.255096,europe_global
6,salt_marsh,1.089227,europe_global
7,cst_lagoon,1.084452,europe_global
8,estuary,0.656618,europe_global
9,saltpan,0.179545,europe_global


In [21]:
# RAMSAR types
europe_ramsar = country_df[['ha_marine_coastal', 'ha_inland', 'ha_man_made']].copy()
europe_ramsar = europe_ramsar.melt(id_vars=[], var_name='label', value_name='value')
europe_ramsar = europe_ramsar.groupby('label').sum().reset_index()
europe_ramsar['value'] = (europe_ramsar['value'] / europe_ramsar['value'].sum()) * 100
europe_ramsar['id'] = 'europe_global'
europe_ramsar['label'] = europe_ramsar['label'].map(ramsar_dict)
print(europe_ramsar['value'].sum())
display(europe_ramsar)

100.0


,label,value,id
0,inland,72.984781,europe_global
1,manmade,10.285767,europe_global
2,coastal,16.729452,europe_global


## Combine datasets

### Wetland types

In [22]:
print(f'Unique labels Europe: {europe_types["label"].nunique()}')
print(f'Unique labels countries: {country_df_long["label"].nunique()}')
print(f'Unique labels PA: {pa_df_long["label"].nunique()}')
print(f'Unique labels river basins: {basin_df_long["label"].nunique()}')

Unique labels Europe: 21
Unique labels countries: 21
Unique labels PA: 21
Unique labels river basins: 21


In [23]:
# Wetland types
wetland_types_df = pd.concat([europe_types, country_df_long, basin_df_long, pa_df_long])
wetland_types_df['value'] = round(wetland_types_df['value'], 2)
print(f'Unique locations: {wetland_types_df["id"].nunique()}')
print(f'Total rows: {wetland_types_df.shape[0]}')
print(f'Unique labels: {wetland_types_df["label"].nunique()}')
display(wetland_types_df.sample(6))


Unique locations: 44771
Total rows: 135923
Unique labels: 21


,label,value,id
54418,rip_broad,13.09,europe_wdpa_9875
192012,rip_mixed,2.86,europe_wdpa_13840
133965,rip_conif,2.63,europe_wdpa_336
853014,marine_low,100.00,europe_wdpa_6697
802020,lake_pond,4.26,europe_wdpa_246
162891,rip_conif,100.00,europe_wdpa_29262


### RAMSAR types

In [24]:
print(f'Unique labels Europe: {europe_ramsar["label"].nunique()}')
print(f'Unique labels countries: {country_df_ramsar_long["label"].nunique()}')
print(f'Unique labels PA: {pa_df_ramsar_long["label"].nunique()}')
print(f'Unique labels river basins: {basin_df_ramsar_long["label"].nunique()}')

Unique labels Europe: 3
Unique labels countries: 3
Unique labels PA: 3
Unique labels river basins: 3


In [25]:
ramsar_types_df = pd.concat([europe_ramsar, country_df_ramsar_long, basin_df_ramsar_long, pa_df_ramsar_long])
ramsar_types_df['value'] = round(ramsar_types_df['value'], 2)
print(f'Unique locations: {ramsar_types_df["id"].nunique()}')
print(f'Total rows: {ramsar_types_df.shape[0]}')
print(f'Unique labels: {ramsar_types_df["label"].nunique()}')
display(ramsar_types_df.sample(6))

Unique locations: 44771
Total rows: 64342
Unique labels: 3


,label,value,id
63013,inland,100.00,europe_wdpa_18470
77332,inland,100.00,europe_wdpa_32789
31325,coastal,100.00,europe_wdpa_31325
21771,coastal,61.50,europe_wdpa_21771
124668,manmade,3.17,europe_wdpa_35582
84969,inland,100.00,europe_wdpa_40426


## Format for indicator-data.json

In [26]:
INDICATOR_DATA_JSON = '../../app-initial-data/indicator-data.json'
LOCATIONS_JSON = '../../app-initial-data/locations.json'

# From 'Styles/Style wetland inventory.qml' of the European Wetlands Atlas
types_color_dict = {
    "rice": "#ca7646",
    "rip_broad": "#4c7300",
    "riparian": "#8109ee",
    "rip_conif": "#737300",
    "rip_mixed": "#895a44",
    "wet_meadow": "#a87000",
    "wet_grass": "#e69800",
    "wet_heath": "#ffaa00",
    "riv_scrub": "#cdaa66",
    "beach_dune": "#ffebaf",
    "inl_marsh": "#00a884",
    "open_mire": "#73dfff",
    "salt_marsh": "#bee8ff",
    "cst_lagoon": "#005ce6",
    "estuary": "#00c5ff",
    "saltpan": "#73b2ff",
    "intertidal": "#d69dbc",
    "watercourse": "#004da8",
    "lake_pond": "#0070ff",
    "marine_low": "#002673",
    "reservoir": "#ff7100",
    "uncert_wet": "#7a668b"
}
# Same colors as the Sahel Ramsar wetland categories
ramsar_color_dict = {
    "coastal": "#3BCC9B",
    "inland": "#1E90FF",
    "manmade": "#FFA500"
}
assert set(label_dict.values()) <= set(types_color_dict), 'Wetland types without a color'
assert set(ramsar_dict.values()) <= set(ramsar_color_dict), 'Ramsar categories without a color'

In [27]:
def to_indicator_data(df, indicator, entry_prefix, data_prefix, labels, colors):
    """One indicator-data entry per location, with the same keys and ID format as the Sahel ones."""
    # 'Europe (total)' of the country data has no location: Europe is europe_global
    missing_id = df['id'].isna()
    print(f'{indicator}: dropping {missing_id.sum()} rows without a location')
    df = df[~missing_id].copy()
    assert df['label'].notna().all(), 'Rows without a label'
    df['order'] = df['label'].map({label: i for i, label in enumerate(labels)})
    df = df.sort_values(['id', 'order'])

    entries = []
    for location, df_location in df.groupby('id', sort=False):
        data = [
            {
                "id": f"{data_prefix}_{location}_{i}",
                "type": "",
                "unit": "%",
                "color": colors[row.label],
                "group": "",
                "label": row.label,
                "value": float(row.value),
                "format": "number"
            }
            for i, row in enumerate(df_location.itertuples())
        ]
        entries.append({
            "id": f"{entry_prefix}-{location}",
            "indicator": indicator,
            "location": location,
            "data": data
        })
    return entries

wetland_types_data = to_indicator_data(
    wetland_types_df, 'europe-wetland-types-get', 'wetland-types', 'wetland_type',
    list(label_dict.values()), types_color_dict)
ramsar_types_data = to_indicator_data(
    ramsar_types_df, 'europe-wetland-types-ramsar', 'wetland-ramsar', 'wetland_ramsar',
    list(ramsar_dict.values()), ramsar_color_dict)
europe_indicator_data = wetland_types_data + ramsar_types_data

ids = [entry['id'] for entry in europe_indicator_data]
assert len(ids) == len(set(ids)), 'Duplicate indicator-data IDs'
print(f'Wetland types entries: {len(wetland_types_data)}')
print(f'Ramsar categories entries: {len(ramsar_types_data)}')
print(json.dumps(ramsar_types_data[0], indent=2))

europe-wetland-types-get: dropping 21 rows without a location
europe-wetland-types-ramsar: dropping 3 rows without a location
Wetland types entries: 44771
Ramsar categories entries: 44771
{
  "id": "wetland-ramsar-europe_adminregion_alb",
  "indicator": "europe-wetland-types-ramsar",
  "location": "europe_adminregion_alb",
  "data": [
    {
      "id": "wetland_ramsar_europe_adminregion_alb_0",
      "type": "",
      "unit": "%",
      "color": "#3BCC9B",
      "group": "",
      "label": "coastal",
      "value": 36.12,
      "format": "number"
    },
    {
      "id": "wetland_ramsar_europe_adminregion_alb_1",
      "type": "",
      "unit": "%",
      "color": "#1E90FF",
      "group": "",
      "label": "inland",
      "value": 57.5,
      "format": "number"
    },
    {
      "id": "wetland_ramsar_europe_adminregion_alb_2",
      "type": "",
      "unit": "%",
      "color": "#FFA500",
      "group": "",
      "label": "manmade",
      "value": 6.38,
      "format": "number"
    

In [ ]:
# The seed skips the indicator data of locations that aren't in locations.json (e.g. the ones outside the
# Europe region, dropped in 05b_europe_locations_preparations.ipynb), so they aren't saved
with open(LOCATIONS_JSON, encoding='utf-8') as f:
    location_ids = {location['id'] for location in json.load(f)}
missing_locations = sorted({entry['location'] for entry in europe_indicator_data} - location_ids)
print(f'Locations not in locations.json: {len(missing_locations)}')
print(missing_locations[:10])

europe_indicator_data = [entry for entry in europe_indicator_data if entry['location'] in location_ids]
print(f'Entries to save: {len(europe_indicator_data)}')

### Save to json

In [33]:
with open(INDICATOR_DATA_JSON, encoding='utf-8') as f:
    seeding_json = json.load(f)

# Updates the existing entries in place, appends the new ones and drops the ones of these
# indicators that are no longer generated, so that the cell can be run again
indicators = {entry['indicator'] for entry in europe_indicator_data}
new_entries = {entry['id']: entry for entry in europe_indicator_data}
updated_ids = set()
merged = []
for entry in seeding_json:
    if entry['id'] in new_entries:
        merged.append(new_entries[entry['id']])
        updated_ids.add(entry['id'])
    elif entry['indicator'] not in indicators:
        merged.append(entry)
merged += [entry for entry in europe_indicator_data if entry['id'] not in updated_ids]
print(f'Updated: {len(updated_ids)}, added: {len(europe_indicator_data) - len(updated_ids)}, '
      f'removed: {len(seeding_json) + len(europe_indicator_data) - len(updated_ids) - len(merged)}')

# Same formatting as the app's export script (JSON.stringify(data, null, 2)), so the existing records don't change
with open(INDICATOR_DATA_JSON, 'w', encoding='utf-8') as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

Updated: 89162, added: 380, removed: 0
